# 🌱 SmartEnergy MAS — Prova de Conceito
### Sistema Multi-Agentes com Q-Learning Cooperativo

**Fazenda Buritis** | Luziânia, Goiás | Dados reais de Janeiro 2025

Este notebook implementa um **Sistema Multi-Agentes (MAS)** para gestão energética de fazendas de grãos.  
Os agentes aprendem via **Q-Learning Cooperativo (IQL)** e são comparados com um baseline de regras heurísticas.

| Agente | Papel | Ações |
|---|---|---|
| 🔋 Armazenamento | Controla a bateria de 24 kWh | Carregar / Manter / Descarregar |
| ⚡ Consumo | Desliga cargas interruptíveis | Nenhum corte / Pivô / Captação / Ambos |
| 🎛️ Gerente de Carga | Define teto de consumo | Conservador 20kW / Moderado 30kW / Liberal 40kW |

**Tarifa azul:** R$0,68/kWh fora de pico → R$1,10/kWh no pico (18h–21h)  
**Objetivo cooperativo:** minimizar custo de energia preservando bateria e processo produtivo.

In [ ]:
!pip install openpyxl --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────
# ⚙️  PARÂMETROS GLOBAIS — ajuste aqui antes de rodar
# ──────────────────────────────────────────────────────────────
CONFIG = {
    # Treinamento
    'n_episodios'     : 2000,    # dias simulados no treino
    'alpha'           : 0.1,     # taxa de aprendizado
    'gamma'           : 0.95,    # fator de desconto futuro
    'epsilon_inicial' : 1.0,     # exploração inicial (100 %)
    'epsilon_final'   : 0.05,    # exploração mínima  (5 %)
    'epsilon_decay'   : 0.9975,  # decaimento por episódio

    # Bateria
    'bateria_cap_kwh' : 24.0,
    'soc_inicial_pct' : 50.0,
    'soc_min_pct'     : 15.0,    # abaixo disso → estado crítico
    'soc_max_pct'     : 95.0,    # acima disso  → para de carregar
    'eficiencia'      : 0.92,

    # Pesos do reward cooperativo
    'w_custo'         : 1.0,
    'pen_soc'         : 10.0,    # violação SOC crítico
    'pen_teto'        : 5.0,     # consumo acima do teto
    'pen_producao'    : 8.0,     # corte de bomba de captação
    'bonus_excedente' : 0.3,     # kWh excedente (crédito)
    'bonus_soc_ok'    : 1.0,     # SOC entre 30 % e 80 %
}

# Tetos de consumo por decisão do Gerente de Carga
TETOS_KW = {0: 20.0, 1: 30.0, 2: 40.0}   # conservador / moderado / liberal

print("✓ Configurações carregadas")
print(f"  Episódios : {CONFIG['n_episodios']}  |  α={CONFIG['alpha']}  |  γ={CONFIG['gamma']}")
print(f"  ε: {CONFIG['epsilon_inicial']} → {CONFIG['epsilon_final']}  (decay={CONFIG['epsilon_decay']})")
print(f"  Bateria   : {CONFIG['bateria_cap_kwh']} kWh  |  SOC crítico < {CONFIG['soc_min_pct']} %")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 📂  CARREGAMENTO DE DADOS — faça upload do Excel quando pedido
# ──────────────────────────────────────────────────────────────
from google.colab import files

print("Selecione: modelo_dados_gestao_energia_fazenda.xlsx")
uploaded = files.upload()
ARQUIVO  = list(uploaded.keys())[0]

def carregar_dados(path):
    xl = pd.read_excel(path, sheet_name=None)

    # Tarifa azul (pico 18h–21h)
    t_row     = xl['Tarifa'][xl['Tarifa']['Tipo_Tarifa'] == 'Tarifa azul'].iloc[0]
    tarifa_24 = np.array([float(t_row[h]) for h in range(24)])

    # Geração: fonte 6 = solar, 7 = eólico
    ger = xl['Geracao'].copy()
    ger['Data'] = pd.to_datetime(ger['Data'])

    # Cargas: 6 = pivô, 7 = captação, 8 = sede, 9 = silo
    car = xl['Cargas'].copy()
    car['Data'] = pd.to_datetime(car['Data'])

    dias = []
    for data in sorted(ger['Data'].unique()):
        g = ger[ger['Data'] == data]
        c = car[car['Data'] == data]

        solar    = g[g['ID_Fonte'] == 6].set_index('Hora')['Energia_Gerada_kWh']
        eolico   = g[g['ID_Fonte'] == 7].set_index('Hora')['Energia_Gerada_kWh']
        pivo     = c[c['ID_equipamento'] == 6].set_index('Hora')['Energia_Consumida_kWh']
        captacao = c[c['ID_equipamento'] == 7].set_index('Hora')['Energia_Consumida_kWh']
        sede     = c[c['ID_equipamento'] == 8].set_index('Hora')['Energia_Consumida_kWh']
        silo     = c[c['ID_equipamento'] == 9].set_index('Hora')['Energia_Consumida_kWh']

        dia = pd.DataFrame({'hora': range(24)})
        dia['solar_kw']    = dia['hora'].map(solar).fillna(0.0)
        dia['eolico_kw']   = dia['hora'].map(eolico).fillna(0.0)
        dia['pivo_kw']     = dia['hora'].map(pivo).fillna(0.0)
        dia['captacao_kw'] = dia['hora'].map(captacao).fillna(0.0)
        dia['sede_kw']     = dia['hora'].map(sede).fillna(0.0)
        dia['silo_kw']     = dia['hora'].map(silo).fillna(0.0)
        dia['data']        = data
        dias.append(dia)

    return dias, tarifa_24

DIAS, TARIFA = carregar_dados(ARQUIVO)

print(f"\n✓ {len(DIAS)} dias carregados  (Janeiro 2025 — Fazenda Buritis)")
print("\nTarifa azul (R$/kWh):")
for h in range(24):
    pico = "  ◄ PICO" if TARIFA[h] > 0.9 else ""
    print(f"  {h:02d}h → R${TARIFA[h]:.6f}{pico}")

# Estatísticas rápidas
sol_med  = np.mean([d['solar_kw'].mean()  for d in DIAS])
cons_med = np.mean([(d['pivo_kw']+d['captacao_kw']+d['sede_kw']+d['silo_kw']).mean() for d in DIAS])
print(f"\nGeração solar média  : {sol_med:.2f} kW/h")
print(f"Consumo total médio  : {cons_med:.2f} kW/h")
print(f"Capacidade bateria   : {CONFIG['bateria_cap_kwh']} kWh")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 🏭  AMBIENTE DE SIMULAÇÃO — FazendaEnergyEnv
#
# Estado discreto (Q-table key): (bucket_hora, bucket_soc,
#                                  bucket_solar, bucket_tarifa)
#   bucket_hora  : hora//6        → 4 valores  (0-5h / 6-11h / 12-17h / 18-23h)
#   bucket_soc   : soc//20        → 5 valores  (0-20 / 20-40 / … / 80-100 %)
#   bucket_solar : low/med/high   → 3 valores  (<5 kW / 5-15 / >15)
#   bucket_tarifa: normal/pico    → 2 valores
#   Total: 4×5×3×2 = 120 estados
#
# Ações por agente:
#   Armazenamento : 0=carregar  1=manter  2=descarregar
#   Consumo       : 0=nada  1=corta_pivo  2=corta_captacao  3=corta_ambos
#   Gerente       : 0=conservador(20kW)  1=moderado(30kW)  2=liberal(40kW)
# ──────────────────────────────────────────────────────────────

class FazendaEnergyEnv:

    def __init__(self, dados_dia, tarifa_24h, cfg):
        self.dados  = dados_dia.reset_index(drop=True)
        self.tarifa = tarifa_24h
        self.cfg    = cfg
        self.reset()

    def reset(self):
        self.hora      = 0
        self.soc       = self.cfg['soc_inicial_pct']
        self.historico = []
        return self._estado()

    def _estado(self):
        r = self.dados.iloc[self.hora]
        return {
            'hora'     : self.hora,
            'soc'      : self.soc,
            'solar_kw' : float(r['solar_kw']),
            'eolico_kw': float(r['eolico_kw']),
            'tarifa'   : self.tarifa[self.hora],
        }

    def discretizar(self, est):
        h = est['hora'] // 6
        s = min(int(est['soc'] / 20), 4)
        g = 0 if est['solar_kw'] < 5 else (1 if est['solar_kw'] < 15 else 2)
        t = 1 if est['tarifa'] > 0.9 else 0
        return (h, s, g, t)

    def step(self, a_arm, a_cons, a_ger):
        est = self._estado()
        r   = self.dados.iloc[self.hora]
        cfg = self.cfg
        cap = cfg['bateria_cap_kwh']

        # ── Gerente: define teto ──────────────
        teto = TETOS_KW[a_ger]

        # ── Geração total ─────────────────────
        geracao = float(r['solar_kw']) + float(r['eolico_kw'])

        # ── Consumo após cortes ───────────────
        fixo     = float(r['sede_kw']) + float(r['silo_kw'])   # não cortável
        pivo     = float(r['pivo_kw'])
        captacao = float(r['captacao_kw'])

        cortes = {0: (0, 0), 1: (pivo, 0), 2: (0, captacao), 3: (pivo, captacao)}
        c_pivo, c_cap = cortes[a_cons]
        corte_producao = c_cap > 0      # captação = bomba principal

        consumo        = min(fixo + (pivo - c_pivo) + (captacao - c_cap), teto)
        teto_excedido  = consumo >= teto * 0.99

        # ── Bateria ───────────────────────────
        soc_kwh     = (self.soc / 100.0) * cap
        soc_min_kwh = (cfg['soc_min_pct'] / 100.0) * cap
        soc_max_kwh = (cfg['soc_max_pct'] / 100.0) * cap
        efic        = cfg['eficiencia']

        if a_arm == 0:       # carregar
            disponivel = max(0.0, geracao - consumo)
            carga      = min(disponivel * efic, soc_max_kwh - soc_kwh)
            soc_kwh   += carga
            bat_delta  = carga
        elif a_arm == 2:     # descarregar
            falta      = max(0.0, consumo - geracao)
            descarga   = min(falta, soc_kwh - soc_min_kwh)
            soc_kwh   -= descarga
            bat_delta  = -descarga
        else:                # manter
            bat_delta  = 0.0

        self.soc    = max(0.0, min(100.0, (soc_kwh / cap) * 100.0))
        soc_critico = self.soc < cfg['soc_min_pct']

        # ── Energia da rede ───────────────────
        descarga_util = abs(bat_delta) if bat_delta < 0 else 0.0
        rede_kwh      = max(0.0, consumo - geracao - descarga_util)
        excedente     = max(0.0, geracao - consumo - max(0.0, bat_delta))
        custo         = rede_kwh * est['tarifa']

        # ── Reward cooperativo ────────────────
        reward = (
            - cfg['w_custo']         * custo
            - cfg['pen_soc']         * float(soc_critico)
            - cfg['pen_teto']        * float(teto_excedido)
            - cfg['pen_producao']    * float(corte_producao)
            + cfg['bonus_excedente'] * excedente
            + cfg['bonus_soc_ok']   * float(30 < self.soc < 80)
        )

        self.historico.append({
            'hora': self.hora, 'soc': self.soc,
            'geracao_kw': geracao, 'consumo_kw': consumo,
            'rede_kwh': rede_kwh, 'excedente': excedente,
            'custo_r': custo, 'tarifa': est['tarifa'], 'reward': reward,
            'a_arm': a_arm, 'a_cons': a_cons, 'a_ger': a_ger,
        })

        self.hora += 1
        done      = self.hora >= 24
        prox      = self._estado() if not done else est
        return prox, reward, done, {'custo': custo, 'rede_kwh': rede_kwh}


print("✓ FazendaEnergyEnv definido")
print(f"  Estados discretos : 4×5×3×2 = {4*5*3*2}")
print(f"  Ações combinadas  : 3×4×3   = {3*4*3}")
print(f"  Resolução temporal: 1 h / timestep   →   24 timesteps / episódio")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 📋  AGENTES HEURÍSTICOS — Baseline sem aprendizado
#     Implementa as regras do documento SmartEnergy MAS v1.1
# ──────────────────────────────────────────────────────────────

class AgentesHeuristicos:

    # ── Agente Financeiro: Índice de Estresse (0-100) ─────────
    def stress_financeiro(self, est):
        t_min  = 0.681282
        t_max  = 1.103868
        s_tar  = (est['tarifa'] - t_min) / (t_max - t_min) * 70.0
        s_bat  = 30.0 if est['soc'] < 20 else (10.0 if est['soc'] < 40 else 0.0)
        return min(100.0, s_tar + s_bat)

    # ── Agente de Armazenamento ───────────────────────────────
    # Regras: carrega com sol, descarrega no pico tarifário
    def armazenamento(self, est):
        soc    = est['soc']
        solar  = est['solar_kw']
        tarifa = est['tarifa']
        if soc > CONFIG['soc_max_pct']:
            return 1                         # cheio → manter
        if soc < CONFIG['soc_min_pct'] + 2:
            return 1                         # crítico → não forçar descarga
        if tarifa > 0.9:
            return 2                         # pico tarifário → descarregar
        if solar > 10 and soc < 80:
            return 0                         # sol alto → carregar
        if solar > 3  and soc < 50:
            return 0
        return 1                             # default: manter

    # ── Agente de Consumo ─────────────────────────────────────
    # Regras: corta cargas por nível de estresse financeiro
    # Silo (processo produtivo) nunca é cortado
    def consumo(self, est, stress):
        if stress > 75:
            return 3                         # corta pivô + captação
        if stress > 50:
            return 2                         # corta só captação (maior carga)
        if stress > 25:
            return 1                         # corta só pivô
        return 0                             # sem corte

    # ── Gerente de Carga ──────────────────────────────────────
    def gerente(self, est, stress):
        if stress > 70:
            return 0                         # conservador (20 kW)
        if stress > 35:
            return 1                         # moderado    (30 kW)
        return 2                             # liberal     (40 kW)


HEURISTICA = AgentesHeuristicos()

def rodar_heuristico(dados_dia, tarifa_24h):
    env = FazendaEnergyEnv(dados_dia, tarifa_24h, CONFIG)
    est = env.reset()
    for _ in range(24):
        stress = HEURISTICA.stress_financeiro(est)
        a_arm  = HEURISTICA.armazenamento(est)
        a_cons = HEURISTICA.consumo(est, stress)
        a_ger  = HEURISTICA.gerente(est, stress)
        est, _, done, _ = env.step(a_arm, a_cons, a_ger)
        if done:
            break
    return env.historico

print("✓ Agentes heurísticos definidos")
# Teste rápido no primeiro dia
h_test = rodar_heuristico(DIAS[0], TARIFA)
custo_test = sum(r['custo_r'] for r in h_test)
print(f"  Teste dia 01/01 → custo heurístico: R${custo_test:.2f}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 🤖  AGENTES Q-LEARNING — IQL Cooperativo
#
#  Cada agente tem Q-table independente (Independent Q-Learning).
#  Todos recebem o mesmo reward cooperativo do ambiente.
#  Isso incentiva o sistema a otimizar o objetivo global,
#  não apenas o comportamento individual de cada agente.
# ──────────────────────────────────────────────────────────────

class AgenteQL:

    def __init__(self, n_acoes, nome, cfg):
        self.n_acoes   = n_acoes
        self.nome      = nome
        self.epsilon   = cfg['epsilon_inicial']
        self.alpha     = cfg['alpha']
        self.gamma     = cfg['gamma']
        self.eps_min   = cfg['epsilon_final']
        self.eps_decay = cfg['epsilon_decay']
        self.q_table   = defaultdict(lambda: np.zeros(n_acoes))
        self.n_updates = 0

    def agir(self, estado_disc, explorando=True):
        if explorando and np.random.random() < self.epsilon:
            return np.random.randint(self.n_acoes)    # exploração aleatória
        return int(np.argmax(self.q_table[estado_disc]))  # política greedy

    def aprender(self, s, a, r, s2, done):
        q_atual  = self.q_table[s][a]
        q_alvo   = r if done else r + self.gamma * np.max(self.q_table[s2])
        self.q_table[s][a] += self.alpha * (q_alvo - q_atual)
        self.n_updates += 1

    def decair_epsilon(self):
        self.epsilon = max(self.eps_min, self.epsilon * self.eps_decay)

    @property
    def n_estados(self):
        return len(self.q_table)


# ── Instanciar os 3 agentes ───────────────────────────────────
AGENTES = {
    'armazenamento': AgenteQL(3, 'Armazenamento', CONFIG),
    'consumo'      : AgenteQL(4, 'Consumo',       CONFIG),
    'gerente'      : AgenteQL(3, 'Gerente',        CONFIG),
}

print("✓ Agentes Q-Learning instanciados")
print(f"  {'Agente':<16} | Ações | ε inicial")
for ag in AGENTES.values():
    print(f"  {ag.nome:<16} |   {ag.n_acoes}   | {ag.epsilon:.2f}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 🏋️  LOOP DE TREINAMENTO
# ──────────────────────────────────────────────────────────────

def treinar(dias, tarifa_24h, agentes, cfg):
    rewards_hist = []
    custos_hist  = []
    n_ep         = cfg['n_episodios']

    for ep in range(n_ep):
        # Sorteia um dia aleatório de janeiro para cada episódio
        dados_dia = dias[np.random.randint(len(dias))]
        env       = FazendaEnergyEnv(dados_dia, tarifa_24h, cfg)
        est       = env.reset()
        s_disc    = env.discretizar(est)

        ep_reward = 0.0
        ep_custo  = 0.0

        for _ in range(24):
            # ── Cada agente age independentemente ────────────
            a_arm  = agentes['armazenamento'].agir(s_disc)
            a_cons = agentes['consumo'].agir(s_disc)
            a_ger  = agentes['gerente'].agir(s_disc)

            prox_est, reward, done, info = env.step(a_arm, a_cons, a_ger)
            s2_disc = env.discretizar(prox_est)

            # ── Atualização cooperativa: mesmo reward ─────────
            agentes['armazenamento'].aprender(s_disc, a_arm,  reward, s2_disc, done)
            agentes['consumo'].aprender(      s_disc, a_cons, reward, s2_disc, done)
            agentes['gerente'].aprender(      s_disc, a_ger,  reward, s2_disc, done)

            s_disc    = s2_disc
            ep_reward += reward
            ep_custo  += info['custo']

        for ag in agentes.values():
            ag.decair_epsilon()

        rewards_hist.append(ep_reward)
        custos_hist.append(ep_custo)

        # Log a cada 200 episódios
        if (ep + 1) % 200 == 0:
            w       = min(100, ep + 1)
            r_med   = np.mean(rewards_hist[-w:])
            c_med   = np.mean(custos_hist[-w:])
            eps     = agentes['armazenamento'].epsilon
            n_est   = agentes['armazenamento'].n_estados
            print(f"Ep {ep+1:>5}/{n_ep}  reward={r_med:>8.2f}  "
                  f"custo=R${c_med:>5.2f}  ε={eps:.3f}  estados={n_est}")

    return rewards_hist, custos_hist


print(f"Iniciando treinamento  ({CONFIG['n_episodios']} episódios × 24 timesteps)...\n")
REWARDS_HIST, CUSTOS_HIST = treinar(DIAS, TARIFA, AGENTES, CONFIG)

print("\n✓ Treinamento concluído!")
for ag in AGENTES.values():
    print(f"  {ag.nome:<16} | {ag.n_estados:>3} estados visitados | {ag.n_updates:>6} updates | ε={ag.epsilon:.3f}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 📈  CURVA DE APRENDIZADO
# ──────────────────────────────────────────────────────────────

def media_movel(arr, w=50):
    return np.convolve(arr, np.ones(w) / w, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Curva de Aprendizado — Q-Learning Cooperativo", fontsize=13, fontweight='bold')

ax1.plot(REWARDS_HIST, alpha=0.2, color='steelblue', lw=0.8)
ax1.plot(media_movel(REWARDS_HIST), color='steelblue', lw=2.2, label='Média móvel (50 ep.)')
ax1.set_xlabel('Episódio'); ax1.set_ylabel('Reward total do dia')
ax1.set_title('Reward por Episódio'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(CUSTOS_HIST, alpha=0.2, color='tomato', lw=0.8)
ax2.plot(media_movel(CUSTOS_HIST), color='tomato', lw=2.2, label='Média móvel (50 ep.)')
ax2.set_xlabel('Episódio'); ax2.set_ylabel('Custo (R$)')
ax2.set_title('Custo de Energia por Episódio'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curva_aprendizado.png', dpi=150, bbox_inches='tight')
plt.show()

c_ini = np.mean(CUSTOS_HIST[:50])
c_fim = np.mean(CUSTOS_HIST[-50:])
print(f"Custo médio (primeiros 50 ep.) : R${c_ini:.2f}")
print(f"Custo médio (últimos  50 ep.)  : R${c_fim:.2f}")
if c_ini > 0:
    print(f"Redução aprendida              : {((c_ini - c_fim) / c_ini * 100):.1f} %")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 🔬  AVALIAÇÃO — Heurístico vs RL em todos os dias de Janeiro
# ──────────────────────────────────────────────────────────────

def rodar_rl(dados_dia, tarifa_24h, agentes):
    env    = FazendaEnergyEnv(dados_dia, tarifa_24h, CONFIG)
    est    = env.reset()
    s_disc = env.discretizar(est)
    for _ in range(24):
        a_arm  = agentes['armazenamento'].agir(s_disc, explorando=False)
        a_cons = agentes['consumo'].agir(s_disc, explorando=False)
        a_ger  = agentes['gerente'].agir(s_disc, explorando=False)
        prox, _, done, _ = env.step(a_arm, a_cons, a_ger)
        s_disc = env.discretizar(prox)
        if done:
            break
    return env.historico

# Avaliar em todos os 31 dias
RES_H = [rodar_heuristico(d, TARIFA) for d in DIAS]
RES_R = [rodar_rl(d, TARIFA, AGENTES) for d in DIAS]

def resumo_mes(historicos):
    custo  = np.mean([sum(h['custo_r']  for h in hist) for hist in historicos])
    rede   = np.mean([sum(h['rede_kwh'] for h in hist) for hist in historicos])
    viols  = np.mean([sum(1 for h in hist if h['soc'] < CONFIG['soc_min_pct']) for hist in historicos])
    reward = np.mean([sum(h['reward']   for h in hist) for hist in historicos])
    return custo, rede, viols, reward

CH, RH, VH, RWH = resumo_mes(RES_H)
CR, RR, VR, RWR = resumo_mes(RES_R)

def delta(a, b):
    return f"{((b - a) / abs(a) * 100):>+.1f} %" if a != 0 else "  n/a"

print(f"{'AVALIAÇÃO — MÉDIA DIÁRIA (Janeiro 2025)':^62}")
print(f"{'═'*62}")
print(f"  {'Métrica':<30} {'Heurístico':>12} {'RL':>10} {'Δ':>8}")
print(f"  {'─'*58}")
print(f"  {'Custo (R$/dia)':<30} {CH:>12.2f} {CR:>10.2f} {delta(CH, CR):>8}")
print(f"  {'kWh da rede / dia':<30} {RH:>12.2f} {RR:>10.2f} {delta(RH, RR):>8}")
print(f"  {'Violações SOC / dia':<30} {VH:>12.2f} {VR:>10.2f}")
print(f"  {'Reward médio / dia':<30} {RWH:>12.2f} {RWR:>10.2f} {delta(RWH, RWR):>8}")
print(f"{'═'*62}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# 📊  VISUALIZAÇÃO DETALHADA — dia onde RL mais se destaca
# ──────────────────────────────────────────────────────────────

difs = [sum(RES_H[i][h]['custo_r'] for h in range(24)) -
        sum(RES_R[i][h]['custo_r'] for h in range(24))
        for i in range(len(DIAS))]
idx_melhor = int(np.argmax(difs))

hist_h = RES_H[idx_melhor]
hist_r = RES_R[idx_melhor]
horas  = list(range(24))
COR_H  = '#e74c3c'
COR_R  = '#27ae60'

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
data_str  = DIAS[idx_melhor]['data'].iloc[0].strftime('%d/%m/%Y')
fig.suptitle(f"Heurístico vs RL  —  {data_str}  (maior diferença de custo)",
             fontsize=13, fontweight='bold')

# ── 1. Custo por hora ─────────────────────────────────────────
ax = axes[0, 0]
ax.bar([h - 0.2 for h in horas], [r['custo_r'] for r in hist_h], 0.4,
       label='Heurístico', color=COR_H, alpha=0.85)
ax.bar([h + 0.2 for h in horas], [r['custo_r'] for r in hist_r], 0.4,
       label='RL', color=COR_R, alpha=0.85)
ax.axvspan(17.5, 20.5, alpha=0.12, color='orange', label='Pico tarifário')
ch_ = sum(r['custo_r'] for r in hist_h)
cr_ = sum(r['custo_r'] for r in hist_r)
ax.set_title(f"Custo por Hora  |  Heur R${ch_:.2f}  →  RL R${cr_:.2f}  ({((cr_-ch_)/ch_*100):+.1f} %)")
ax.set_xlabel('Hora'); ax.set_ylabel('R$'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# ── 2. SOC da bateria ─────────────────────────────────────────
ax = axes[0, 1]
ax.plot(horas, [r['soc'] for r in hist_h], 'o-', color=COR_H, label='Heurístico', ms=4)
ax.plot(horas, [r['soc'] for r in hist_r], 's-', color=COR_R, label='RL', ms=4)
ax.axhline(CONFIG['soc_min_pct'], color='red',  ls='--', alpha=0.6, label=f"SOC crítico ({CONFIG['soc_min_pct']} %)")
ax.axhline(80,                    color='blue', ls='--', alpha=0.4, label='SOC ótimo (80 %)')
ax.axvspan(17.5, 20.5, alpha=0.12, color='orange')
ax.set_title('Estado de Carga da Bateria (%)'); ax.set_xlabel('Hora'); ax.set_ylabel('SOC (%)')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0, 105)

# ── 3. Geração vs Consumo ─────────────────────────────────────
ax = axes[1, 0]
ger_vals = [r['geracao_kw'] for r in hist_r]
ax.fill_between(horas, ger_vals, alpha=0.3, color='gold', label='Geração (solar+eólico)')
ax.plot(horas, [r['consumo_kw'] for r in hist_h], 'o-', color=COR_H, label='Consumo Heurístico', ms=4)
ax.plot(horas, [r['consumo_kw'] for r in hist_r], 's-', color=COR_R, label='Consumo RL', ms=4)
ax.axvspan(17.5, 20.5, alpha=0.12, color='orange')
ax.set_title('Geração vs Consumo (kW)'); ax.set_xlabel('Hora'); ax.set_ylabel('kW')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── 4. Mapa de decisões do RL ─────────────────────────────────
ax = axes[1, 1]
cores_arm  = ['#3498db', '#95a5a6', '#e67e22']  # carregar/manter/descarregar
cores_cons = ['#27ae60', '#f1c40f', '#e67e22', '#e74c3c']
cores_ger  = ['#e74c3c', '#e67e22', '#27ae60']

for h in range(24):
    ax.barh(2.5, 1, left=h, height=0.75, color=cores_arm[hist_r[h]['a_arm']],  alpha=0.9)
    ax.barh(1.5, 1, left=h, height=0.75, color=cores_cons[hist_r[h]['a_cons']], alpha=0.9)
    ax.barh(0.5, 1, left=h, height=0.75, color=cores_ger[hist_r[h]['a_ger']],  alpha=0.9)

ax.axvspan(17.5, 20.5, alpha=0.12, color='orange')
ax.set_yticks([0.5, 1.5, 2.5])
ax.set_yticklabels(['Gerente', 'Consumo', 'Armaz.'])
ax.set_xlabel('Hora'); ax.set_title('Decisões dos Agentes RL por Hora'); ax.set_xlim(0, 24)

leg_arm  = [mpatches.Patch(color=c, label=l) for c, l in
            zip(cores_arm, ['Carregar', 'Manter', 'Descarregar'])]
ax.legend(handles=leg_arm, loc='upper left', fontsize=7, title='Armaz.', title_fontsize=7)

plt.tight_layout()
plt.savefig('comparativo_dia.png', dpi=150, bbox_inches='tight')
plt.show()


## 🌦️ Análise de Cenários
Compara heurístico vs RL em três condições distintas usando dados reais de janeiro.

| Cenário | Critério |
|---|---|
| ☁️ Dia nublado | Menor geração solar+eólica total |
| ☀️ Dia ensolarado | Maior geração solar+eólica total |
| ⚡ Alto consumo | Maior razão consumo/geração (pior caso energético) |

In [ ]:
# ──────────────────────────────────────────────────────────────
# 🌦️  ANÁLISE DE CENÁRIOS
# ──────────────────────────────────────────────────────────────

ger_dia  = [d['solar_kw'].sum() + d['eolico_kw'].sum() for d in DIAS]
cons_dia = [(d['pivo_kw'] + d['captacao_kw'] + d['sede_kw'] + d['silo_kw']).sum() for d in DIAS]
razao    = [cons_dia[i] / max(ger_dia[i], 0.1) for i in range(len(DIAS))]

CENARIOS = {
    '☁️  Nublado'     : int(np.argmin(ger_dia)),
    '☀️  Ensolarado'  : int(np.argmax(ger_dia)),
    '⚡ Alto Consumo' : int(np.argmax(razao)),
}

print(f"{'ANÁLISE DE CENÁRIOS':^68}")
print(f"{'═'*68}")
print(f"  {'Cenário':<18} {'Dia':>5} {'Geração':>10} {'Custo Heur.':>12} {'Custo RL':>10} {'Δ%':>7}")
print(f"  {'─'*64}")

resultados_cenarios = {}
for nome, idx in CENARIOS.items():
    h_h = rodar_heuristico(DIAS[idx], TARIFA)
    h_r = rodar_rl(DIAS[idx], TARIFA, AGENTES)
    c_h = sum(r['custo_r'] for r in h_h)
    c_r = sum(r['custo_r'] for r in h_r)
    resultados_cenarios[nome] = (h_h, h_r, c_h, c_r, idx)
    data_s = DIAS[idx]['data'].iloc[0].strftime('%d/%m')
    delt   = ((c_r - c_h) / c_h * 100) if c_h > 0 else 0
    print(f"  {nome:<18} {data_s:>5} {ger_dia[idx]:>9.1f} kWh {c_h:>10.2f} R$ {c_r:>8.2f} R$ {delt:>+6.1f} %")

print(f"{'═'*68}")

# ── Plot 3 cenários ───────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("Cenários: Heurístico vs RL", fontsize=14, fontweight='bold')
horas = list(range(24))

for row, (nome, (h_h, h_r, c_h, c_r, idx)) in enumerate(resultados_cenarios.items()):
    data_s = DIAS[idx]['data'].iloc[0].strftime('%d/%m/%Y')
    delt   = ((c_r - c_h) / c_h * 100) if c_h > 0 else 0

    ax1 = axes[row, 0]
    ax1.bar([h - 0.2 for h in horas], [r['custo_r'] for r in h_h], 0.4,
            color=COR_H, alpha=0.85, label='Heurístico')
    ax1.bar([h + 0.2 for h in horas], [r['custo_r'] for r in h_r], 0.4,
            color=COR_R, alpha=0.85, label='RL')
    ax1.axvspan(17.5, 20.5, alpha=0.12, color='orange')
    ax1.set_title(f"{nome}  {data_s}  |  R${c_h:.2f} → R${c_r:.2f}  ({delt:+.1f} %)")
    ax1.set_ylabel('R$'); ax1.legend(fontsize=7); ax1.grid(axis='y', alpha=0.3)

    ax2 = axes[row, 1]
    ax2.plot(horas, [r['soc'] for r in h_h], 'o-', color=COR_H, label='Heurístico', ms=3)
    ax2.plot(horas, [r['soc'] for r in h_r], 's-', color=COR_R, label='RL', ms=3)
    ax2.axhline(CONFIG['soc_min_pct'], color='red',  ls='--', alpha=0.5)
    ax2.axhline(80,                    color='blue', ls='--', alpha=0.3)
    ax2.axvspan(17.5, 20.5, alpha=0.12, color='orange')
    ax2.set_title(f"SOC da Bateria — {nome}"); ax2.set_ylabel('SOC (%)'); ax2.set_ylim(0, 105)
    ax2.legend(fontsize=7); ax2.grid(alpha=0.3)
    if row == 2:
        ax1.set_xlabel('Hora'); ax2.set_xlabel('Hora')

plt.tight_layout()
plt.savefig('cenarios.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ──────────────────────────────────────────────────────────────
# 🏁  RESUMO FINAL DA PoC
# ──────────────────────────────────────────────────────────────

d_custo = ((CR - CH) / CH * 100) if CH > 0 else 0
d_rede  = ((RR - RH) / RH * 100) if RH > 0 else 0
d_rew   = ((RWR - RWH) / abs(RWH) * 100) if RWH != 0 else 0
sinal_c = "🟢 ECONOMIA" if d_custo < 0 else "🔴 AUMENTO"
sinal_r = "🟢 MELHORA"  if d_rew   > 0 else "🔴 PIORA"

print()
print("╔══════════════════════════════════════════════════════════════╗")
print("║          RESUMO FINAL — SmartEnergy MAS PoC                 ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Episódios treinados  : {CONFIG['n_episodios']:<5}                              ║")
print(f"║  Dias avaliados       : {len(DIAS):<5} (Janeiro 2025)                 ║")
print(f"║  Tamanho bateria      : {CONFIG['bateria_cap_kwh']} kWh                          ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Custo médio heurístico : R$ {CH:>6.2f} / dia                    ║")
print(f"║  Custo médio RL         : R$ {CR:>6.2f} / dia                    ║")
print(f"║  Variação de custo      : {sinal_c} {d_custo:>+.1f} %                   ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  kWh da rede / dia (heur) : {RH:>6.2f}                          ║")
print(f"║  kWh da rede / dia (RL)   : {RR:>6.2f}  ({d_rede:>+.1f} %)              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Violações SOC heurístico : {VH:.2f} / dia                      ║")
print(f"║  Violações SOC RL         : {VR:.2f} / dia                      ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Reward médio heur. : {RWH:>8.2f}                              ║")
print(f"║  Reward médio RL    : {RWR:>8.2f}   {sinal_r} {d_rew:>+.1f} %            ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║  Q-tables aprendidas:                                        ║")
for ag in AGENTES.values():
    line = f"║    {ag.nome:<16}  {ag.n_estados:>3} estados  {ag.n_updates:>6} updates"
    print(f"{line:<63}║")
print("╚══════════════════════════════════════════════════════════════╝")
